# 04 — Ensemble Blending

Demonstrates combining the Polymarket signal with auxiliary signals into a single calibrated forecast:

1. **Signal inputs** — PM, sentiment, fundamentals, options, Markov
2. **Weight blending** — PM scaled by avg quality, others fixed weights, normalised to sum=1
3. **Variance** — combines market-level uncertainty + sentiment variance + 20% model buffer
4. **Confidence interval** — 95% CI using 1.96σ with horizon-based floor
5. **Quality score** — 0-100 composite (breadth, avg quality, precision, diversity, whale absence)
6. **End-to-end forecast** — current price → forecast price → action signal

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.prices import fetch_historical_prices
from research.data.polymarket import fetch_polymarket_markets
from research.models.ensemble import (
    MarketInput,
    OtherSignals,
    compute_polymarket_signal,
    compute_ensemble,
    compute_variance,
    compute_ci,
    compute_quality_score,
    score_to_grade,
    run_ensemble,
)
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
    compute_markov_forecast,
)

## 1. Load Data

Fetch prices, Polymarket markets, and compute a Markov signal to use as one of the ensemble inputs.

In [ ]:
TICKER = "BTC"
CURRENT_PRICE = 78000.0
HORIZON = 7

# Fetch prices
prices = fetch_historical_prices(TICKER, days=120)
prices['returns'] = prices['close'].pct_change()
returns = prices['returns'].dropna().values

# Fetch Polymarket markets
markets_df = fetch_polymarket_markets("bitcoin", limit=10)

# Build MarketInput objects
market_inputs = []
for _, row in markets_df.iterrows():
    market_inputs.append(MarketInput(
        question=row['question'],
        probability=row['probability'],
        volume24h_usd=row['volume_24h'],
        age_days=int(row['age_days']) if pd.notna(row['age_days']) else None,
        signal_tier='macro',
        delta_yes=0.06,
        delta_no=-0.04,
    ))

# Compute Markov signal
regimes = classify_regime_series(returns)
P = estimate_transition_matrix(regimes, decay_rate=0.97)
forecast = compute_markov_forecast(P, regimes[-1], HORIZON)

def compute_regime_up_rates(regimes, returns, horizon, decay_rate=0.97):
    counts = {s: {'up': 0.0, 'total': 0.0} for s in ['bull', 'bear', 'sideways']}
    max_start = min(len(regimes), len(returns)) - horizon
    for i in range(max_start):
        regime = regimes[i]
        cum_return = sum(returns[i+1:i+1+horizon])
        weight = decay_rate ** (max_start - 1 - i)
        counts[regime]['total'] += weight
        if cum_return > 0:
            counts[regime]['up'] += weight
    return {
        s: counts[s]['up'] / counts[s]['total'] if counts[s]['total'] > 0 else 0.5
        for s in ['bull', 'bear', 'sideways']
    }

up_rates = compute_regime_up_rates(regimes, returns, HORIZON)
markov_p_up = sum(forecast[s] * up_rates[s] for s in ['bull', 'bear', 'sideways'])
markov_return = markov_p_up - 0.5  # centred around zero

print(f"Polymarket markets: {len(market_inputs)}")
print(f"Markov P(up):       {markov_p_up:.4f}")
print(f"Markov return:      {markov_return:.4f}")

## 2. Polymarket Signal

In [ ]:
pm = compute_polymarket_signal(market_inputs)
print(f"PM Signal:    {pm['signal']:.4f}  ({pm['signal']*100:.2f}%)")
print(f"Avg Quality:  {pm['avg_quality']:.4f}")
print(f"Warnings:     {len(pm['warnings'])}")

## 3. Ensemble Blending

Base weights:
- PM = 0.40 × pm_avg_quality
- Sentiment = 0.20
- Fundamental = 0.25
- Options = 0.15
- Markov = 0.20

Weights are normalised to sum to 1 across available signals.

In [ ]:
others = OtherSignals(
    sentiment_score=0.3,       # mildly bullish
    fundamental_return=0.02,   # +2% expected over horizon
    options_skew=0.5,          # bullish put-call skew
    markov_return=markov_return,
    horizon_days=HORIZON,
)

ensemble = compute_ensemble(pm['signal'], pm['avg_quality'], others)

print(f"Forecast return: {ensemble['forecast_return']:.4f}  ({ensemble['forecast_return']*100:.2f}%)")
print("\nWeights:")
for name, w in ensemble['weights'].items():
    print(f"  {name:12s}: {w:.4f} ({w*100:.1f}%)")

In [ ]:
# Pie chart of weights
fig, ax = plt.subplots(figsize=(8, 8))
weights = ensemble['weights']
ax.pie(
    weights.values(),
    labels=[f"{k}\n{v*100:.1f}%" for k, v in weights.items()],
    autopct='',
    startangle=90,
)
ax.set_title('Ensemble Signal Weights')
plt.tight_layout()
plt.show()

## 4. Variance and Confidence Interval

Variance components:
- PM market-level uncertainty: `p_adj × (1-p_adj) × spread²`
- Sentiment variance: `(sent_weight × 0.04)²`
- Model-uncertainty buffer: `× 1.2`

Floor: `σ_floor = 0.10 × √(horizon/252)` prevents implausibly tight CIs.

In [ ]:
sigma = compute_variance(
    market_inputs,
    ensemble['weights'].get('pm', 0),
    ensemble['weights'].get('sentiment', 0),
    others.sentiment_score,
)

# Apply horizon floor
import math
horizon_frac = max(1, HORIZON) / 252
sigma_floor = 0.10 * math.sqrt(horizon_frac)
sigma = max(sigma_floor, sigma)

forecast_price = CURRENT_PRICE * (1 + ensemble['forecast_return'])
ci = compute_ci(forecast_price, sigma)

print(f"Forecast price:  ${forecast_price:,.2f}")
print(f"Sigma:           {sigma:.4f}")
print(f"95% CI:          [${ci['low']:,.2f}, ${ci['high']:,.2f}]")
print(f"Current price:   ${CURRENT_PRICE:,.2f}")
print(f"CI width:        ${ci['high'] - ci['low']:,.2f}  ({100*(ci['high']-ci['low'])/CURRENT_PRICE:.1f}% of price)")

In [ ]:
# Visualise the forecast distribution (Gaussian approximation)
import scipy.stats as stats

x = np.linspace(ci['low'] * 0.8, ci['high'] * 1.2, 500)
y = stats.norm.pdf(x, loc=forecast_price, scale=sigma * CURRENT_PRICE)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(x, y, alpha=0.3, label='Forecast distribution')
ax.axvline(CURRENT_PRICE, color='black', linestyle='--', label=f'Current: ${CURRENT_PRICE:,.0f}')
ax.axvline(forecast_price, color='blue', linewidth=2, label=f'Forecast: ${forecast_price:,.0f}')
ax.axvline(ci['low'], color='red', linestyle=':', label=f'95% CI lower: ${ci["low"]:,.0f}')
ax.axvline(ci['high'], color='red', linestyle=':', label=f'95% CI upper: ${ci["high"]:,.0f}')
ax.set_xlabel('Price')
ax.set_ylabel('Density')
ax.set_title(f'{TICKER} {HORIZON}-Day Forecast Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Quality Score

Composite score (0-100) based on:
- **Breadth** (30%): number of markets, max 5
- **Avg quality** (25%): mean market quality weight
- **Precision** (20%): tight CI → higher score
- **Signal diversity** (15%): fraction of 4 possible signals present
- **Whale absence** (10%): penalty for markets with price spikes

In [ ]:
signals_with_data = sum([
    1 if market_inputs else 0,
    1 if (others.sentiment_score is not None) else 0,
    1 if (others.fundamental_return is not None) else 0,
    1 if (others.options_skew is not None) else 0,
    1 if (others.markov_return is not None) else 0,
])

whale_count = sum(1 for m in market_inputs if m.price_spike_detected)

quality = compute_quality_score(
    market_inputs,
    pm['avg_quality'],
    sigma,
    signals_with_data,
    whale_count,
)
grade = score_to_grade(quality)

print(f"Quality score:  {quality}/100")
print(f"Grade:          {grade}")
print(f"Whale count:    {whale_count}")
print(f"Signals used:   {signals_with_data}/5")

In [ ]:
# Decompose quality score
s1 = 30 * min(len(market_inputs), 5) / 5
s2 = 25 * pm['avg_quality']
s3 = 20 * max(0, 1 - sigma / 0.20)
s4 = 15 * (signals_with_data / 4)
s5 = 10 * (1 - whale_count / len(market_inputs)) if market_inputs else 0

components = pd.DataFrame({
    'Component': ['Breadth', 'Avg Quality', 'Precision', 'Diversity', 'Whale Absence'],
    'Score': [s1, s2, s3, s4, s5],
    'Max': [30, 25, 20, 15, 10],
})
components['%'] = (components['Score'] / components['Max'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(components))
ax.barh(x, components['Score'], color='steelblue')
ax.set_yticks(x)
ax.set_yticklabels(components['Component'])
ax.set_xlabel('Score')
ax.set_title(f'Quality Score Decomposition: {quality}/100 (Grade {grade})')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. End-to-End Ensemble Forecast

Run the full pipeline in one call.

In [ ]:
result = run_ensemble(
    current_price=CURRENT_PRICE,
    markets=market_inputs,
    others=others,
)

print(f"=== {TICKER} {HORIZON}-Day Forecast ===")
print(f"Forecast return:   {result.forecast_return*100:+.2f}%")
print(f"Forecast price:    ${result.forecast_price:,.2f}")
print(f"95% CI:            [${result.ci_low95:,.2f}, ${result.ci_high95:,.2f}]")
print(f"Sigma:             {result.sigma:.4f}")
print(f"Quality score:     {result.quality_score}/100  (Grade {result.quality_grade})")
print(f"PM signal:         {result.pm_signal*100:.2f}%  (weight: {result.pm_effective_weight*100:.1f}%)")
print(f"Avg market qual:   {result.avg_market_quality:.4f}")
print(f"Warnings:          {len(result.warnings)}")
for w in result.warnings[:3]:
    print(f"  - {w}")

## 7. Sensitivity: Missing Signals

What happens when we remove auxiliary signals one by one?

In [ ]:
scenarios = {
    "All signals": OtherSignals(sentiment_score=0.3, fundamental_return=0.02, options_skew=0.5, markov_return=markov_return, horizon_days=HORIZON),
    "No sentiment": OtherSignals(fundamental_return=0.02, options_skew=0.5, markov_return=markov_return, horizon_days=HORIZON),
    "No fundamentals": OtherSignals(sentiment_score=0.3, options_skew=0.5, markov_return=markov_return, horizon_days=HORIZON),
    "No options": OtherSignals(sentiment_score=0.3, fundamental_return=0.02, markov_return=markov_return, horizon_days=HORIZON),
    "No Markov": OtherSignals(sentiment_score=0.3, fundamental_return=0.02, options_skew=0.5, horizon_days=HORIZON),
    "PM only": OtherSignals(horizon_days=HORIZON),
}

results = []
for name, sigs in scenarios.items():
    r = run_ensemble(CURRENT_PRICE, market_inputs, sigs)
    results.append({
        'scenario': name,
        'return_pct': r.forecast_return * 100,
        'ci_low': r.ci_low95,
        'ci_high': r.ci_high95,
        'quality': r.quality_score,
        'grade': r.quality_grade,
    })

sens_df = pd.DataFrame(results)
print(sens_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(sens_df))

# Error bars = CI width
ci_width = sens_df['ci_high'] - sens_df['ci_low']
ax.errorbar(
    x,
    sens_df['return_pct'],
    yerr=ci_width / 2 / CURRENT_PRICE * 100,  # convert to pct
    fmt='o',
    markersize=8,
    capsize=5,
    linewidth=2,
)
ax.set_xticks(x)
ax.set_xticklabels(sens_df['scenario'], rotation=15, ha='right')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Forecast Return (%)')
ax.set_title(f'{TICKER}: Ensemble Sensitivity to Missing Signals')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()